# Colab: LoRA fine-tuning for Component 1

**J26-SE-325** — run this on Colab, not on the development laptop.

Two independent reasons the local machine cannot do this:

1. **No CUDA.** The dev machine has an AMD Radeon 610M, so `torch` there is the CPU build.
   Fine-tuning a 200M-parameter model on CPU is hours per run.
2. **~24 KB/s network.** The foundation checkpoints are ~821 MB each — roughly 10 hours
   apiece. Three attempts left only zero-byte `.incomplete` placeholders.

Colab solves both: a free T4 and a fast link.

## What this notebook does

| Part | Output |
|---|---|
| A | LoRA-fine-tune TimesFM and/or Chronos-Bolt → adapter weights (a few MB) |
| B | SFT-fine-tune a small Llama on the tool-calling trajectories → the Phase 5c agent |

Both produce artifacts small enough to bring home over a slow link. **Download the adapters,
not the base models** — that asymmetry is the entire point of using LoRA here.

## Before running

Runtime → Change runtime type → **T4 GPU**.

In [ ]:
!nvidia-smi
import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Set Runtime -> Change runtime type -> T4 GPU'

## Setup

Clone the repo, install only what fine-tuning needs. `timesfm` is pinned at **2.0.2** —
there is no `2.5` on PyPI; "2.5" is the model generation exposed as
`TimesFM_2p5_200M_torch`. Installing `2.5` fails outright.

In [ ]:
REPO_URL = 'https://github.com/SE-Y4S1/J26-SE-325.git'   # update if the remote moves
COMPONENT = 'J26-SE-325/backend/Portfolio-Optimization'

import os
if not os.path.exists('J26-SE-325'):
    !git clone --depth 1 $REPO_URL
%cd $COMPONENT
!ls

In [ ]:
# Deliberately NOT `pip install -r requirements.txt`: that file pins the CPU torch build
# (+cpu), which would replace Colab's CUDA torch and silently drop you back to CPU.
!pip install -q "timesfm[torch]==2.0.2" chronos-forecasting peft accelerate \
                pandas-ta-openbb pyyaml yfinance pyarrow 2>&1 | tail -3

import torch
print('torch after install:', torch.__version__, '| cuda:', torch.cuda.is_available())
assert torch.cuda.is_available(), 'CUDA torch was replaced — reinstall the CUDA build'

## Part A — LoRA fine-tune the foundation forecasters

Builds the same feature table the local pipeline uses, driven by the committed
`configs/resolved_universe.yaml` so each symbol contributes exactly the history its own
criteria justified. Nothing about the windows is re-decided here.

In [ ]:
import sys, warnings, logging
from datetime import date
from pathlib import Path

warnings.filterwarnings('ignore')
logging.basicConfig(level=logging.INFO, format='%(levelname)s %(message)s')
sys.path.insert(0, str(Path.cwd()))

import pandas as pd, yaml
from data.ingestion import bars_to_frame, fetch_ohlcv
from data.schema import AssetClass
from features.feature_store import add_targets, build_feature_table
from features.technical import compute_universe_indicators

CLASSES = {'equity': AssetClass.EQUITY, 'etf': AssetClass.ETF, 'forex': AssetClass.FOREX}
resolved = yaml.safe_load(Path('configs/resolved_universe.yaml').read_text())['symbols']

bars = {}
for symbol, entry in sorted(resolved.items()):
    rows = fetch_ohlcv(symbol, CLASSES[entry['asset_class']],
                       date.fromisoformat(entry['history_start']),
                       date.fromisoformat(entry['history_end']))
    if rows:
        bars[symbol] = bars_to_frame(rows)

HORIZON = 5
features = add_targets(build_feature_table(bars, compute_universe_indicators(bars)), horizon=HORIZON)
print(f'{len(bars)} symbols | {len(features):,} feature rows | '
      f"{features.timestamp.min().date()} -> {features.timestamp.max().date()}")

In [ ]:
from forecasting.finetune_lora import LoRAConfig, finetune

# Larger than the CPU defaults — a T4 can afford it.
config = LoRAConfig(r=16, lora_alpha=32, epochs=20, batch_size=64, lr=5e-5)

adapters = {}
for model_name in ('timesfm', 'chronos_bolt'):
    try:
        adapters[model_name] = finetune(
            model_name, features, horizon=HORIZON, config=config,
            output_dir=Path(f'artifacts/checkpoints/{model_name}-lora-h{HORIZON}'),
        )
        print(f'OK  {model_name} -> {adapters[model_name]}')
    except Exception as exc:
        # One model failing must not cost the other; RQ1 reports whichever succeeded.
        print(f'FAILED {model_name}: {type(exc).__name__}: {exc}')

adapters

### Train the hybrid head

The residual head is the part the TAF actually commits to ("hybrid TimesFM + LSTM/MLP").
It is zero-initialised, so the hybrid starts *exactly* at the base model's accuracy and any
improvement is attributable to the covariates rather than to a lucky init.

In [ ]:
from forecasting.base import get_forecaster
from forecasting.hybrid_model import HybridConfig, HybridForecaster
from forecasting.residual_head import ResidualHeadConfig

base_name = 'timesfm' if 'timesfm' in adapters else 'chronos_bolt'
base = get_forecaster(base_name, adapter_path=str(adapters[base_name]))

hybrid = HybridForecaster(base, HybridConfig(
    base_model=base_name,
    head=ResidualHeadConfig(hidden_size=128, num_layers=2, epochs=40),
    window=60,
))
hybrid.fit(features, horizon=HORIZON)

decomposed = hybrid.decompose(features, horizon=HORIZON)
print('base / residual / final (means):')
for q in (10, 50, 90):
    print(f'  p{q}: {decomposed[f"base_p{q}"].mean():+.5f}  '
          f'{decomposed[f"residual_p{q}"].mean():+.5f}  {decomposed[f"final_p{q}"].mean():+.5f}')

### RQ1 — the full comparison

This is the table the local machine cannot produce. Walk-forward, no look-ahead, every model
on identical folds.

In [ ]:
from evaluation.backtest import BacktestConfig, run_walk_forward
from evaluation.metrics import forecast_metrics

config_bt = BacktestConfig(train_window_days=1095, test_window_days=180,
                           step_days=180, embargo_days=HORIZON)

rows = []
for name in ['baseline_lstm', 'timesfm', 'chronos_bolt', 'hybrid']:
    try:
        preds = run_walk_forward(features, name, horizon=HORIZON, config=config_bt)
        valid = preds.dropna(subset=['target_return'])
        rows.append({'model': name, 'n_folds': int(preds.fold.nunique()),
                     'n_predictions': len(valid),
                     **forecast_metrics(valid.target_return.to_numpy(),
                                        valid[['p10','p50','p90']].to_numpy(), (0.1,0.5,0.9))})
    except Exception as exc:
        rows.append({'model': name, 'status': f'failed: {exc}'})

rq1 = pd.DataFrame(rows)
Path('artifacts/results').mkdir(parents=True, exist_ok=True)
rq1.to_csv('artifacts/results/rq1_forecast.csv', index=False)
rq1

**Reading the table.** Compare on **pinball loss**, not MAE — MAE only scores the median and
says nothing about whether the p10–p90 band is honest, which is what Phase 5a's CVaR consumes.
Also check `calibration_error_*`: the local baseline showed all three quantiles sitting below
nominal (a uniform level bias from trailing-window training through a bull market). If the
hybrid closes that gap, say so explicitly — it is the clearest evidence the residual head
earns its place.

## Part B — SFT the tool-calling agent

Trains the model that replaces `agent/reference_agent.py`. The dataset is generated locally
by `agent/trajectory_generation.py`; every trajectory in it passed `enforce_grounding`, so
the model learns to route numbers through the optimizer rather than inventing them.

**Upload `artifacts/trajectories/*.jsonl` to the Colab session before running this.**

In [ ]:
import json, glob

paths = sorted(glob.glob('artifacts/trajectories/*.jsonl'))
assert paths, 'No trajectories. Run agent/trajectory_generation.py locally and upload the JSONL.'

records = [json.loads(line) for p in paths for line in open(p, encoding='utf-8')]
print(f'{len(records)} trajectories from {len(paths)} file(s)')

# Non-negotiable: every record must have called the grounded tool. Training on even a few
# ungrounded transcripts teaches the model that inventing a number is sometimes acceptable,
# which is the exact failure the whole design exists to prevent.
GROUNDING_TOOL = 'run_fuzzy_ga_withdrawal'
bad = [r for r in records if GROUNDING_TOOL not in r['metadata']['tool_calls']]
assert not bad, f'{len(bad)} ungrounded trajectories — regenerate, do not train on these'
print('all trajectories grounded')

In [ ]:
!pip install -q transformers trl datasets bitsandbytes 2>&1 | tail -2

BASE_MODEL = 'meta-llama/Llama-3.2-3B-Instruct'   # gated: accept the licence on HF first
# Ungated alternative if the licence is a blocker:
# BASE_MODEL = 'Qwen/Qwen2.5-3B-Instruct'

from huggingface_hub import notebook_login
notebook_login()      # skip if the chosen base is ungated

In [ ]:
from datasets import Dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def to_text(record):
    """Render a trajectory through the model's own chat template.

    Using apply_chat_template rather than hand-formatting matters: the tool-call special
    tokens must match what the model emits at inference, or the fine-tune teaches a format
    the runtime cannot parse.
    """
    messages = []
    for m in record['messages']:
        entry = {'role': m['role'], 'content': m.get('content') or ''}
        if m.get('tool_calls'):
            entry['content'] = json.dumps(m['tool_calls'])
        messages.append(entry)
    return tokenizer.apply_chat_template(messages, tokenize=False)

dataset = Dataset.from_dict({'text': [to_text(r) for r in records]}).train_test_split(test_size=0.1, seed=42)
print(dataset)
print(dataset['train'][0]['text'][:600])

In [ ]:
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL, torch_dtype=torch.bfloat16, device_map='auto', load_in_4bit=True,
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset['train'],
    eval_dataset=dataset['test'],
    peft_config=LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
        target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj'],
    ),
    args=SFTConfig(
        output_dir='artifacts/checkpoints/agent-sft',
        num_train_epochs=3, per_device_train_batch_size=2,
        gradient_accumulation_steps=8, learning_rate=2e-4,
        logging_steps=20, eval_strategy='epoch', save_strategy='epoch',
        bf16=True, max_length=2048, report_to='none',
    ),
)
trainer.train()
trainer.save_model('artifacts/checkpoints/agent-sft')

## Bring the results home

Download **adapters only** — a few MB against ~821 MB of base weights. On a 24 KB/s link
that is the difference between a minute and a day.

In [ ]:
import shutil
from pathlib import Path

shutil.make_archive('component1_adapters', 'zip', 'artifacts/checkpoints')
size_mb = Path('component1_adapters.zip').stat().st_size / 1e6
print(f'component1_adapters.zip  {size_mb:.1f} MB')

from google.colab import files
files.download('component1_adapters.zip')

# Also take the RQ1 table — it is the dissertation result, and it is only kilobytes.
files.download('artifacts/results/rq1_forecast.csv')

## Back on the local machine

```bash
unzip component1_adapters.zip -d artifacts/checkpoints/
```

Then register the checkpoints so `/portfolio/*` responses carry a resolvable
`model_version` — Component 3 anchors provenance on it, and an unregistered adapter is
invisible to the on-chain bridge:

```python
from datetime import date
from pathlib import Path
from forecasting.model_registry import register

register('hybrid-timesfm', Path('artifacts/checkpoints/timesfm-lora-h5'),
         train_start=date(2011, 1, 3), train_end=date(2025, 12, 31),
         metrics={'val_pinball': 0.0},   # copy the real number from the RQ1 table
         activate=True)
```

For the agent, serve the SFT adapter through Ollama (or vLLM) and point
`OLLAMA_MODEL` at it. The tools, schemas and `enforce_grounding` validator stay exactly as
they are — only the driver changes, which is what the reference agent was built to prove.